# MAMA-MIA — 09: Kattalashtirish bo'yicha doza-javob

## Nega bu notebook boshqacha

Birinchi loyiha quyidagicha edi: V0 bashoratlarini fizik fazoga qaytarib
Dice'ni qayta o'lchash. **Bu printsipial ishlamaydi.** Dice qayta
masshtablashga invariant:

```
Dice = 2|P∩G| / (|P|+|G|)
```

Uch hadning hammasi bir xil koeffitsientga ko'payadi — nisbat o'zgarmaydi.
Sintetik sinovda o'lchandi: bitta maska juftligi, x1.0 → 0.5118,
x2.0 → 0.5118, x3.0 → 0.5100, anizotrop (3.0, 3.0, 1.76) → 0.5178.
Qolgan tebranish faqat diskretizatsiya shovqini.

Demak kuzatilgan effekt **metrika artefakti emas**. Mexanizm modelda:

> 2D U-Net chegara xatosini taxminan **pikselda** doimiy qiladi.
> V0 har bir manbaga boshqa mm/piksel beradi. Shuning uchun **fizik**
> chegara xatosi — va u bilan Dice — har bir manbaning kattalashtirish
> koeffitsientiga bog'liq ravishda tizimli o'zgaradi.

## Bu notebook nimani o'lchaydi

Kattalashtirish **bevosita boshqariladi**, manbalararo taqqoslash kerak emas.

V2 ma'lumoti (1.5 mm izotrop, fizik masshtab saqlangan) olinadi va
**barcha manbaga bir xil** k koeffitsienti qo'llanadi:

| k | effektiv tekislik qadami |
|---|---|
| 0.5 | 3.0 mm (o'sma kichik ko'rinadi) |
| 1.0 | 1.5 mm (o'zgarishsiz) |
| 2.0 | 0.75 mm (o'sma katta ko'rinadi) |

Har bir k uchun chegara xatosi **ikki birlikda** o'lchanadi (ASSD):

- `assd_px` — pikselda
- `assd_mm` — millimetrda

## Oldindan qayd etilgan taxmin

(Oldingi taxminim noto'g'ri chiqqani uchun bu safar yozib qo'yaman.)

1. Dice k bilan **o'sadi**.
2. `assd_px` k bo'yicha taxminan **doimiy** qoladi.
3. `assd_mm` taxminan **1/k** bo'yicha kamayadi.

2 va 3 bajarilsa — mexanizm bevosita o'lchangan bo'ladi: tarmoq pikselda
ishlaydi, metrika esa fizik o'lchamda baholanadi, V0 esa bu ikkisi
orasidagi kursni har bir manbaga boshqacha belgilaydi.

Agar `assd_px` ham k bilan o'zgarsa — tushuntirish to'liq emas va
maqolada shunday yoziladi.

## Cheklovlar (ochiq aytiladi, har biri o'lchanadi)

**1. Ko'rish maydoni.** Piksel soni qat'iy 192 bo'lgani uchun k ni
o'zgartirish ko'rish maydonini ham o'zgartiradi (k kichik → maydon katta).
Bu ikkisini ajratish mumkin emas — lekin V0 ning o'zida ham aynan shu
bog'lanish bor, shuning uchun tajriba V0 ni to'g'ri modellashtiradi.
O'lchanadi: `border_frac` (o'smaning chetga tegib qolgan ulushi) va
`gt_vox / k²` nisbati (1 dan uzoqlashsa — qirqish bor).

**2. Interpolyatsiya silliqlashi.** `order=1` qayta namunalash shovqinni
silliqlaydi, shuning uchun k bilan tasvir sigmasi ham o'zgaradi
(sinovda: k=0.5 → 0.33, k=1 → 1.00, k=2 → 0.63). Kirishning global
sigma masshtabini birinchi BatchNorm asosan yutadi, shuning uchun bu
birinchi tartibda zararsiz — lekin spektral o'zgarish qoladi.
O'lchanadi: `img_std`.

Ikkala omil ham natija jadvalida alohida chiqadi, ya'ni doza-javob
ularga tayanganmi yoki yo'qmi — ko'rinib turadi.

## Qurilma

Qurilmani o'zi aniqlaydi: CUDA (Kaggle) → MPS (Apple Silicon) → CPU.
Har bir run tugashi bilan JSON'ga yoziladi, shuning uchun to'xtasa
qolgan joydan davom etadi — Kaggle'da boshlab Mac'da tugatish mumkin.



## 1. Sozlamalar

In [ ]:
import os, gc, json, time, random, glob, collections
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom as ndzoom, binary_erosion
from scipy.ndimage import distance_transform_edt as edt

MAGS     = [0.5, 1.0, 2.0]        # kattalashtirish koeffitsienti
HELD_OUT = ["NACT", "DUKE"]       # eng katta effekt va nazorat
SEEDS    = [42, 1]                # juftlashgan statistika uchun 2 seed

IMG_SIZE, BATCH, LR = 192, 16, 3e-4
EPOCHS, EPOCH_SLICES, EVAL_EVERY = 50, 12000, 10
VAL_FRAC, POS_FRAC = 0.2, 0.7
BASE_SPACING = 1.5                # V2 ma'lumotining qadami

# --- qurilma: CUDA -> MPS -> CPU ---
if torch.cuda.is_available():
    dev, AMP, NUM_WORKERS, PIN = torch.device("cuda"), True, 2, True
    name = torch.cuda.get_device_name(0)
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    dev, AMP, NUM_WORKERS, PIN = torch.device("mps"), False, 0, False
    name = "Apple Silicon (MPS)"
else:
    dev, AMP, NUM_WORKERS, PIN = torch.device("cpu"), False, 0, False
    name = "CPU"
print(f"Qurilma: {dev}  {name}   AMP={AMP}  workers={NUM_WORKERS}")

# --- chiqish papkasi: Kaggle yoki mahalliy ---
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CKPT = os.path.join(OUT, "magsweep")
os.makedirs(CKPT, exist_ok=True)
print("Natijalar:", os.path.abspath(CKPT))
print(f"\nProtokol: k={MAGS}, fold={HELD_OUT}, seed={SEEDS} "
      f"→ {len(MAGS)*len(HELD_OUT)*len(SEEDS)} run")

## 2. Ma'lumot

V2 (`V2_iso_scale`) — 1.5 mm izotrop, fizik masshtab saqlangan.
Kaggle'da Input'dan, mahalliy ishlaganda `DATA_DIR` dan topiladi.

In [ ]:
DATA_DIR = os.environ.get("MAMAMIA_V2", "")   # mahalliy ishlatsangiz shuni bering

cand = sorted({os.path.dirname(p) for p in
               glob.glob("/kaggle/input/**/V2_iso_scale/mamamia_*.h5", recursive=True)})
if DATA_DIR:
    cand = [DATA_DIR] + cand
assert cand, ("V2_iso_scale topilmadi. Kaggle'da variantlar notebook'ini Input "
              "qiling; mahalliy ishlatsangiz MAMAMIA_V2 muhit o'zgaruvchisini "
              "papkaga qo'ying.")
DATA = cand[0]
print("V2:", DATA)

def case_of(k):
    return k[:-2] if k.endswith(("_L","_R")) else k

INDEX = []
for p in sorted(glob.glob(os.path.join(DATA, "mamamia_*.h5"))):
    src = os.path.basename(p)[len("mamamia_"):-len(".h5")]
    with h5py.File(p, "r") as h:
        for key in h:
            mk = h[key]["msk"]
            assert mk.shape[0] == IMG_SIZE == mk.shape[1], \
                f"{key}: kutilgan {IMG_SIZE}x{IMG_SIZE}, topilgan {mk.shape}"
            nz = np.where(np.asarray(mk).any(axis=(0,1)))[0]
            INDEX.append({"src":src,"file":p,"key":key,"case":case_of(key),
                          "n":int(mk.shape[2]),"pos":nz.tolist()})
print(f"{len(INDEX)} namuna, {len({r['case'] for r in INDEX})} holat")
print(collections.Counter(r["src"] for r in INDEX))

## 3. Kattalashtirish va o'lchov

Normallash kattalashtirishdan **oldin** bajariladi — shunda k faqat
geometriyaga ta'sir qiladi, intensivlik statistikasiga emas.
Qirqish/to'ldirish massiv **markaziga** nisbatan (o'smaga nisbatan emas —
aks holda ground truth sizadi).

In [ ]:
def fit_center(a, pad_value):
    """Massiv markaziga nisbatan IMG_SIZE ga qirqish yoki to'ldirish."""
    H, W = a.shape
    if H == IMG_SIZE and W == IMG_SIZE:
        return a
    out = np.full((IMG_SIZE, IMG_SIZE), pad_value, a.dtype)
    hs, ws = min(H, IMG_SIZE), min(W, IMG_SIZE)
    sy, sx = (H - hs) // 2, (W - ws) // 2
    dy, dx = (IMG_SIZE - hs) // 2, (IMG_SIZE - ws) // 2
    out[dy:dy+hs, dx:dx+ws] = a[sy:sy+hs, sx:sx+ws]
    return out


def magnify(img, msk, k):
    """img z-score'langan float32, msk float32/uint8. k=1 bo'lsa tegilmaydi."""
    if abs(k - 1.0) < 1e-9:
        return img, msk
    i = ndzoom(img, k, order=1, mode="nearest")
    m = ndzoom(msk, k, order=0, mode="nearest")
    return fit_center(i, 0.0), fit_center(m, 0.0)      # 0.0 = z-score o'rtasi


def dice(a, b):
    s = a.sum() + b.sum()
    return (2.0 * np.logical_and(a, b).sum() / s) if s else float("nan")


def surf(M):
    return M & ~binary_erosion(M, iterations=1, border_value=0)


def assd(P, G, sampling):
    """O'rtacha simmetrik sirt masofasi, berilgan qadamda."""
    sp, sg = surf(P), surf(G)
    if not sp.any() or not sg.any():
        return float("nan")
    dg = edt(~sg, sampling=sampling)
    dp = edt(~sp, sampling=sampling)
    return float((dg[sp].mean() + dp[sg].mean()) / 2.0)


# --- Sinov: k faqat geometriyani o'zgartiradimi ---
_r = np.random.default_rng(0)
_i = _r.normal(size=(IMG_SIZE, IMG_SIZE)).astype(np.float32)
_m = np.zeros((IMG_SIZE, IMG_SIZE), np.float32); _m[80:110, 85:115] = 1
print("k ning o'smaga ta'siri (bitta kesim, 30x30 piksel = 45x45 mm):")
for k in MAGS:
    ii, mm = magnify(_i, _m, k)
    print(f"  k={k:<4} o'sma {int(mm.sum()):>6} piksel "
          f"(kutilgan ~{int(30*30*k*k):>6})   "
          f"tasvir o'rtacha {ii.mean():+.3f}  sigma {ii.std():.3f}")

# aynan mos maskada ASSD = 0 bo'lishi kerak
_G = np.zeros((40, 40, 10), bool); _G[10:30, 10:30, 2:8] = True
assert assd(_G, _G, (1.5, 1.5, 1.5)) == 0.0, "ASSD mos maskada nol emas"
# 1 voksel qalinlashtirish ~ 1 voksel masofa berishi kerak
from scipy.ndimage import binary_dilation
_P = binary_dilation(_G, iterations=1)
_a_px = assd(_P, _G, (1.0, 1.0, 1.0))
_a_mm = assd(_P, _G, (1.5, 1.5, 1.5))
print(f"\nASSD tekshiruvi (1 voksel qalinlashtirish): "
      f"{_a_px:.3f} piksel = {_a_mm:.3f} mm   nisbat {_a_mm/_a_px:.3f} (kutilgan 1.5)")
assert abs(_a_mm / _a_px - 1.5) < 0.01, "ASSD qadamga chiziqli bog'lanmadi"

## 4. Dataset va model

In [ ]:
NORM = {}

class SliceDS(Dataset):
    def __init__(self, items, train, k, epoch_len=None):
        self.items, self.train, self.k = items, train, k
        self.h = {}
        if train:
            self.epoch_len = epoch_len or EPOCH_SLICES
            self.pos = [(i,z) for i,r in enumerate(items) for z in r["pos"]]
            self.all = [(i,z) for i,r in enumerate(items) for z in range(r["n"])]
        else:
            self.flat = [(i,z) for i,r in enumerate(items) for z in range(r["n"])]
    def __len__(self): return self.epoch_len if self.train else len(self.flat)
    def _f(self,p):
        if p not in self.h: self.h[p]=h5py.File(p,"r")
        return self.h[p]
    def _st(self,rec):
        key=rec["key"]
        if key not in NORM:
            g=self._f(rec["file"])[key]; st=max(1,g["img"].shape[2]//8)
            sub=np.asarray(g["img"][:,:,::st],dtype=np.float32)
            lo,hi=np.percentile(sub,[1,99])
            if hi<=lo: lo,hi=float(sub.min()),float(sub.max())+1e-3
            c=np.clip(sub,lo,hi)
            NORM[key]=(float(lo),float(hi),float(c.mean()),float(c.std()+1e-6))
        return NORM[key]
    def _get(self,i,z):
        rec=self.items[i]; g=self._f(rec["file"])[rec["key"]]
        img=np.asarray(g["img"][:,:,z],dtype=np.float32)
        msk=np.asarray(g["msk"][:,:,z],dtype=np.float32)
        lo,hi,mu,sd=self._st(rec)
        img=((np.clip(img,lo,hi)-mu)/sd).astype(np.float32)   # normallash OLDIN
        return magnify(img, msk, self.k)                      # keyin kattalashtirish
    def __getitem__(self,i):
        if self.train:
            pool=self.pos if (random.random()<POS_FRAC and self.pos) else self.all
            j,z=pool[random.randrange(len(pool))]
            img,msk=self._get(j,z)
            if random.random()<0.5: img,msk=img[::-1].copy(),msk[::-1].copy()
            if random.random()<0.3: img=img*random.uniform(0.9,1.1)+random.uniform(-0.1,0.1)
        else:
            j,z=self.flat[i]; img,msk=self._get(j,z)
        return torch.from_numpy(np.ascontiguousarray(img))[None], \
               torch.from_numpy(np.ascontiguousarray(msk))[None]
    def __del__(self):
        for f in getattr(self,"h",{}).values():
            try: f.close()
            except Exception: pass

def block(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(True))
class UNet(nn.Module):
    def __init__(self, ch=(32,64,128,256,512)):
        super().__init__()
        self.enc=nn.ModuleList(); c=1
        for k in ch[:-1]: self.enc.append(block(c,k)); c=k
        self.bottom=block(ch[-2],ch[-1])
        self.up=nn.ModuleList(); self.dec=nn.ModuleList()
        for k in reversed(ch[:-1]):
            self.up.append(nn.ConvTranspose2d(k*2,k,2,stride=2)); self.dec.append(block(k*2,k))
        self.head=nn.Conv2d(ch[0],1,1)
    def forward(self,x):
        sk=[]
        for e in self.enc: x=e(x); sk.append(x); x=F.max_pool2d(x,2)
        x=self.bottom(x)
        for u,d,s in zip(self.up,self.dec,reversed(sk)): x=d(torch.cat([u(x),s],1))
        return self.head(x)

bce=nn.BCEWithLogitsLoss()
def dice_loss(l,y,eps=1.0):
    p=torch.sigmoid(l); n=2*(p*y).sum((1,2,3))+eps; d=p.sum((1,2,3))+y.sum((1,2,3))+eps
    return 1-(n/d).mean()
def criterion(l,y): return 0.5*bce(l,y)+0.5*dice_loss(l,y)
print(f"model: {sum(p.numel() for p in UNet().parameters())/1e6:.2f}M parametr")

## 5. Ishga tushirish

In [ ]:
def split_cases(tc, seed, frac=VAL_FRAC):
    cs = sorted(tc); rng = random.Random(seed); rng.shuffle(cs)
    nv = max(1, int(len(cs) * frac))
    return set(cs[nv:]), set(cs[:nv])


def _fwd(model, x):
    if AMP:
        with torch.autocast("cuda", enabled=True):
            return model(x)
    return model(x)


@torch.no_grad()
def val_dice(model, items, k, bs=32):
    model.eval(); ds = SliceDS(items, False, k); vals = []
    for i, rec in enumerate(items):
        inter = union = 0.0
        for s in range(0, rec["n"], bs):
            zz = list(range(s, min(s + bs, rec["n"])))
            xs, ys = zip(*(ds._get(i, z) for z in zz))
            x = torch.from_numpy(np.stack(xs))[:, None].to(dev)
            y = torch.from_numpy(np.stack(ys))[:, None].to(dev)
            p = (torch.sigmoid(_fwd(model, x)) > 0.5).float()
            inter += (p * y).sum().item(); union += p.sum().item() + y.sum().item()
        vals.append(2 * inter / union if union else np.nan)
    del ds
    return float(np.nanmean(vals))


@torch.no_grad()
def evaluate(model, items, k, thr=0.5, bs=32):
    """Dice + ASSD ikki birlikda."""
    model.eval(); ds = SliceDS(items, False, k)
    # k dan keyin tekislik qadami, z o'zgarmaydi
    sp_mm = (BASE_SPACING / k, BASE_SPACING / k, BASE_SPACING)
    sp_px = (1.0, 1.0, k)          # z, tekislik pikselida
    out = {}
    for i, rec in enumerate(items):
        P = np.zeros((IMG_SIZE, IMG_SIZE, rec["n"]), bool)
        G = np.zeros((IMG_SIZE, IMG_SIZE, rec["n"]), bool)
        for s in range(0, rec["n"], bs):
            zz = list(range(s, min(s + bs, rec["n"])))
            xs, ys = zip(*(ds._get(i, z) for z in zz))
            x = torch.from_numpy(np.stack(xs))[:, None].to(dev)
            p = (torch.sigmoid(_fwd(model, x)) > thr).float()
            P[:, :, zz] = p.squeeze(1).permute(1, 2, 0).cpu().numpy() > 0.5
            G[:, :, zz] = np.stack(ys, axis=-1) > 0.5
        # --- chalkashtiruvchi omillarni o'lchash ---
        # 1) intensivlik: interpolyatsiya shovqinni silliqlaydi
        zc = rec["n"] // 2
        istd = float(ds._get(i, zc)[0].std())
        # 2) qirqish: o'smaning chetga tegib qolgan ulushi
        b = 0
        if G.any():
            b = int(G[0].sum() + G[-1].sum() + G[:,0].sum() + G[:,-1].sum())
        out[rec["key"]] = {"dice": dice(P, G),
                           "assd_px": assd(P, G, sp_px),
                           "assd_mm": assd(P, G, sp_mm),
                           "gt_vox": int(G.sum()), "pred_vox": int(P.sum()),
                           "img_std": istd,
                           "border_frac": (b / max(int(G.sum()), 1))}
    del ds
    return out


def run(held_out, k, seed):
    tag = f"k{k:g}__{held_out}__s{seed}"
    done = os.path.join(CKPT, tag + ".json")
    if os.path.exists(done):
        print(f"[o'tkazildi] {tag}"); return json.load(open(done))

    tr_set, va_set = split_cases({r["case"] for r in INDEX if r["src"] != held_out}, seed)
    tr = [r for r in INDEX if r["case"] in tr_set]
    va = [r for r in INDEX if r["case"] in va_set]
    te = [r for r in INDEX if r["src"] == held_out]
    assert not any(r["src"] == held_out for r in tr + va), "chetdagi manba sizdi!"
    print(f"\n=== k={k}  chetda {held_out}  seed {seed} ===")
    print(f"  o'qitish {len(tr)}  validatsiya {len(va)}  tashqi {len(te)}")

    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    if dev.type == "cuda": torch.cuda.manual_seed_all(seed)
    model = UNet().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    dl = DataLoader(SliceDS(tr, True, k, EPOCH_SLICES), batch_size=BATCH, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=PIN, drop_last=True,
                    persistent_workers=NUM_WORKERS > 0)

    best, best_state = -1.0, None
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); tot = n = 0
        for x, y in dl:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            loss = criterion(_fwd(model, x), y)
            if AMP:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            tot += loss.item() * x.size(0); n += x.size(0)
        sched.step()
        if (ep + 1) % EVAL_EVERY == 0 or ep == EPOCHS - 1:
            vd = val_dice(model, va, k); star = ""
            if vd > best:
                best, star = vd, " *"
                best_state = {kk: v.detach().clone() for kk, v in model.state_dict().items()}
            print(f"  ep {ep:>2}  loss {tot/max(n,1):.4f}  val {vd:.4f}{star}  "
                  f"{time.time()-t0:.0f}s", flush=True)

    model.load_state_dict(best_state)
    per = evaluate(model, te, k)
    g = lambda key: float(np.nanmean([v[key] for v in per.values()]))
    res = {"k": k, "held_out": held_out, "seed": seed, "n_test": len(te),
           "spacing_mm": BASE_SPACING / k,
           "dice": g("dice"), "assd_px": g("assd_px"), "assd_mm": g("assd_mm"),
           "gt_vox": g("gt_vox"), "img_std": g("img_std"),
           "border_frac": g("border_frac"), "internal_val": best,
           "per_case": per, "minutes": round((time.time()-t0)/60, 1)}
    json.dump(res, open(done, "w"), indent=2)
    print(f"  Dice {res['dice']:.4f}   ASSD {res['assd_px']:.3f} px = "
          f"{res['assd_mm']:.3f} mm   o'sma {res['gt_vox']:.0f} voksel   "
          f"[{res['minutes']:.0f} daq]")
    del model, opt, dl, best_state; gc.collect()
    if dev.type == "cuda": torch.cuda.empty_cache()
    return res


results = []
t_all = time.time(); total = len(MAGS)*len(HELD_OUT)*len(SEEDS); done_n = 0
for seed in SEEDS:
    for ho in HELD_OUT:
        for k in MAGS:
            results.append(run(ho, k, seed)); done_n += 1
            el = (time.time()-t_all)/60
            print(f"  [{done_n}/{total}]  {el:.0f} daq, "
                  f"qolgan ~{el/done_n*(total-done_n):.0f} daq", flush=True)

## 6. Natija

In [ ]:
COLS = ("held_out","k","spacing_mm","seed","n_test","dice",
        "assd_px","assd_mm","gt_vox","img_std","border_frac",
        "internal_val","minutes")
df = pd.DataFrame([{c: r[c] for c in COLS} for r in results]).sort_values(
        ["held_out","k"])
df.to_csv(os.path.join(OUT, "magsweep.csv"), index=False)
display(df.round(4))

rows = []
for r in results:
    for key, v in r["per_case"].items():
        rows.append({"k": r["k"], "held_out": r["held_out"], "seed": r["seed"],
                     "case": key, **v})
pc = pd.DataFrame(rows)
pc.to_csv(os.path.join(OUT, "magsweep_percase.csv"), index=False)
print(f"\nmagsweep_percase.csv: {len(pc)} qator")

from scipy import stats
print("\n" + "="*74)
print("DOZA-JAVOB: kattalashtirish -> Dice va chegara xatosi")
print("="*74)
for ho in HELD_OUT:
    s = df[df.held_out == ho].sort_values("k")
    if s.empty: continue
    print(f"\n  chetda {ho}:")
    print(f"    {'k':>5} {'qadam':>8} {'Dice':>8} {'ASSD px':>9} {'ASSD mm':>9} "
          f"{'o`sma vox':>10}")
    for _, r in s.iterrows():
        print(f"    {r.k:>5.2f} {r.spacing_mm:>7.2f}m {r.dice:>8.4f} "
              f"{r.assd_px:>9.3f} {r.assd_mm:>9.3f} {r.gt_vox:>10.0f}")

    base = s[s.k == 1.0]
    if len(base):
        b = base.iloc[0]
        print(f"    k=1 ga nisbatan:  ", end="")
        for _, r in s.iterrows():
            if r.k == 1.0: continue
            print(f"k={r.k:g}: Dice {r.dice-b.dice:+.4f}  "
                  f"ASSD_px {r.assd_px/b.assd_px:.2f}x  "
                  f"ASSD_mm {r.assd_mm/b.assd_mm:.2f}x   ", end="")
        print()

print("\n" + "="*74)
print("CHALKASHTIRUVCHI OMILLAR — o'lchangan, yashirilmagan")
print("="*74)
for ho in HELD_OUT:
    s = df[df.held_out == ho].sort_values("k")
    if s.empty: continue
    b = s[s.k == 1.0]
    print(f"\n  chetda {ho}:")
    print(f"    {'k':>5} {'tasvir sigma':>13} {'o`sma/k^2':>11} {'chetga tegish':>14}")
    for _, r in s.iterrows():
        # o'sma k^2 ga proporsional bo'lishi kerak; chetlanish = qirqish
        ratio = (r.gt_vox / (r.k ** 2) / b.iloc[0].gt_vox) if len(b) else float("nan")
        print(f"    {r.k:>5.2f} {r.img_std:>13.3f} {ratio:>11.3f} {r.border_frac:>14.4f}")
    print("    (o'sma/k^2 ≈ 1 bo'lsa qirqish yo'q; sigma o'zgarishi "
          "interpolyatsiya silliqlashi)")

print("""
  Izoh: kirishning global sigma masshtabini birinchi BatchNorm asosan
  yutadi, shuning uchun bu chalkashtiruvchi birinchi tartibda zararsiz.
  Lekin spektral o'zgarish qoladi — maqolada cheklov sifatida yoziladi.""")

print("\n  TAXMINLARNI TEKSHIRISH (holat darajasida, juftlashgan):")
for ho in HELD_OUT:
    sub = pc[pc.held_out == ho]
    if sub.empty: continue
    print(f"\n    chetda {ho}:")
    r1, p1 = stats.spearmanr(sub["k"], sub["dice"])
    print(f"      1) Dice k bilan o'sadimi?   rho = {r1:+.3f}  p = {p1:.2e}"
          f"   {'TASDIQ' if (r1 > 0.1 and p1 < 0.05) else 'RAD'}")
    r2, p2 = stats.spearmanr(sub["k"], sub["assd_px"])
    print(f"      2) assd_px doimiymi?        rho = {r2:+.3f}  p = {p2:.2e}"
          f"   {'TASDIQ' if abs(r2) < 0.15 else 'RAD'}")
    r3, p3 = stats.spearmanr(sub["k"], sub["assd_mm"])
    print(f"      3) assd_mm 1/k bo'yichami?  rho = {r3:+.3f}  p = {p3:.2e}"
          f"   {'TASDIQ' if (r3 < -0.1 and p3 < 0.05) else 'RAD'}")
    # juftlashgan: har bir holat uchun k=1 va eng katta k
    w = sub.pivot_table(index="case", columns="k", values="dice")
    if 1.0 in w and max(MAGS) in w:
        a = w[[1.0, max(MAGS)]].dropna()
        if len(a) > 5:
            t = stats.wilcoxon(a[1.0], a[max(MAGS)])
            print(f"      k=1 vs k={max(MAGS):g} juftlashgan Dice: "
                  f"{a[1.0].mean():.4f} -> {a[max(MAGS)].mean():.4f}  "
                  f"({a[max(MAGS)].mean()-a[1.0].mean():+.4f})  p = {t.pvalue:.2e}  n={len(a)}")

print("""
  O'QISH:
  1 + 2 + 3 tasdiqlansa — mexanizm bevosita o'lchangan: tarmoq pikselda
  ishlaydi, metrika fizik o'lchamda baholaydi, V0 esa bu kursni har bir
  manbaga boshqacha belgilaydi. Manbalararo V0 Dice'lari bir xil
  vazifaning o'lchovi emas.

  2 rad etilsa — tarmoq kattalashtirishga o'zini moslashtiradi va
  tushuntirish to'liq emas. Bu ham natija; maqolada shunday yoziladi.""")

## 7. Keyingi qadam

`magsweep.csv` va `magsweep_percase.csv` ni yuboring.

Mahalliy ishlatish uchun:

```bash
export MAMAMIA_V2=/yo'l/V2_iso_scale
jupyter nbconvert --to notebook --execute 09_magsweep.ipynb --inplace
```

Natijalar har run'dan keyin `magsweep/` ichiga yoziladi — to'xtatib,
keyin davom etish mumkin (Kaggle'da boshlab Mac'da tugatish ham).